In [ ]:
# MIT License

# Copyright (c) 2024 Multifidelity Roofline Analysis

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.


from GenZ import decode_moddeling, prefill_moddeling

from GenZ.system import System
import pandas as pd
from GenZ.analyse_model import get_model_df, get_summary_table

from GenZ import create_inference_mamba_prefix_model, create_inference_mamba_decode_model

In [ ]:
# This will reload the imported modules (e.g. get_decode_model_characterstics) every time you execute the jupyter cells, so that you don't need to restart the notebook after updating the source codes.
%load_ext autoreload
%autoreload 2   

In [ ]:
system = System(frequency=1000 , flops=2000, off_chip_mem_size=(80*1024), compute_efficiency=0.8, memory_efficiency=0.8,
                    offchip_mem_bw=3500, bits='int8', external_mem_bw=128, interchip_link_bw=256, interchip_link_latency=2)

In [ ]:
################################################################################################## #
### Model Characterization Calculation
################################################################################################## #
model_prefill = create_inference_mamba_prefix_model(input_sequence_length=1000,output_gen_tokens = 0 ,
                                    name='tiiuae/falcon-mamba-7b')

B = 1
model_df = get_model_df(model_prefill, system, batch_size=B,  model_characterstics = True)
summary_table = get_summary_table(model_df, model_characterstics = True)


In [ ]:
from GenZ.Models.utils import parse_einsum_expression

In [ ]:
L = 1000
S = 16
F = 4096
logL= 3
parse_einsum_expression('lbfs,lbs->lbf', (logL,'b',F,S), (L,'b',S), (logL,'b',F))

In [ ]:
display(model_df)
display(summary_table)

In [ ]:
import matplotlib.pyplot as plt

# Initialize a dictionary to store Op Intensity for each Op Type
op_intensity_data = {op_idx: [] for op_idx in range(len(model_df))}

# Vary B from 1 to 128
batch_sizes = [1,2,3,4,8,16,32,64,128]
for B in batch_sizes:
    model_df = get_model_df(model_prefill, system, batch_size=B, model_characterstics=True)
    for op_idx in range(len(model_df)):
        subset = model_df.loc[op_idx]
        op_intensity_data[op_idx].append(subset['Op Intensity'])

# Plotting
plt.figure(figsize=(10, 6))
for op_type, intensities in op_intensity_data.items():
    plt.plot(batch_sizes, intensities, marker='o', linestyle='-', label=op_type)

plt.xlabel('Batch Size (B)')
plt.ylabel('Op Intensity')
plt.title('Op Intensity of Each Op Type as a Function of B')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
################################################################################################## #
### Model Characterization Calculation
################################################################################################## #
model_decode = create_inference_mamba_decode_model(input_sequence_length=10000,output_gen_tokens = 0 ,
                                    name='tiiuae/falcon-mamba-7b')

B = 1
model_df = get_model_df(model_decode, system, batch_size=B,  model_characterstics = True)
summary_table = get_summary_table(model_df, model_characterstics = True)


In [ ]:
display(model_df)
display(summary_table)

In [ ]:
import matplotlib.pyplot as plt

# Initialize a dictionary to store Op Intensity for each Op Type
op_intensity_data = {op_idx: [] for op_idx in range(len(model_df))}

# Vary B from 1 to 128
batch_sizes = [1,2,3,4,8,16,32,64,128]
for B in batch_sizes:
    model_df = get_model_df(model_decode, system, batch_size=B, model_characterstics=True)
    for op_idx in range(len(model_df)):
        subset = model_df.loc[op_idx]
        op_intensity_data[op_idx].append(subset['Op Intensity'])

# Plotting
plt.figure(figsize=(10, 6))
for op_type, intensities in op_intensity_data.items():
    plt.plot(batch_sizes, intensities, marker='o', linestyle='-', label=op_type)

plt.xlabel('Batch Size (B)')
plt.ylabel('Op Intensity')
plt.title('Op Intensity of Each Op Type as a Function of B')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model_df